# Structure-Validated De Novo Binder Design

End-to-end pipeline: **BoltzGen -> LigandMPNN -> BoltzFold**

1. **BoltzGen** generates de novo backbones conditioned on a target
2. **LigandMPNN** designs sequences for each backbone
3. **BoltzFold** refolds each designed sequence to validate structure and binding

In [ ]:
import os

import torch
from loguru import logger

from evedesign.system import System, Protein
from evedesign.models.boltzgen import BoltzGenGenerator
from evedesign.models.mpnn import LigandMPNN
from evedesign.models.boltzfold import BoltzFoldTransformer

## Step 1: Define target and binder

This notebook uses **1G13** (human GM2 activator protein, chain A) as the target - the canonical test case from BoltzGen's `vanilla_protein/1g13prot.yaml` example. The target CIF is downloaded from RCSB on first run and cached locally so we can extract its sequence (and visualize later).

System layout:

- **Target** (`rep=<1G13 chain A sequence>` plus `structures`): the crystal structure is attached, so BoltzGen receives a `file:` entry and designs against the experimental conformation. Without `structures` it would auto-generate an MSA and fold the target itself, which is the right choice only when no structure is available.
- **Binder** (`rep=None`, `min_length=80`, `max_length=140`): the de novo binder. No starting sequence, no starting backbone - BoltzGen invents both. The length range matches BoltzGen's vanilla binder convention.

To swap in a different target, change `TARGET_PDB_ID` and `TARGET_CHAIN` below.

In [ ]:
import urllib.request
from pathlib import Path

import biotite.structure as struc

from evedesign.sequence import Sequences
from evedesign.structure import Structure, StructureFile
from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

# Target configuration 
TARGET_PDB_ID = "1G13"          # GM2 activator protein
TARGET_CHAIN = "A"
BINDER_MIN_LENGTH = 80           # BoltzGen vanilla default
BINDER_MAX_LENGTH = 140

# Pipeline sizing
NUM_BACKBONES = 50               # backbones from BoltzGen
NUM_SEQUENCES_PER_BACKBONE = 20   # MPNN sequences per backbone

# Download target CIF (cached locally)
CACHE_DIR = Path("./target_cache")
CACHE_DIR.mkdir(exist_ok=True)
cif_path = CACHE_DIR / f"{TARGET_PDB_ID.lower()}.cif"

if not cif_path.exists():
    url = f"https://files.rcsb.org/download/{TARGET_PDB_ID}.cif"
    print(f"Downloading {TARGET_PDB_ID} from RCSB...")
    urllib.request.urlretrieve(url, cif_path)
    print(f"  -> cached at {cif_path}")
else:
    print(f"Using cached {TARGET_PDB_ID} from {cif_path}")

# Extract target sequence from chain A 
sf = StructureFile(str(cif_path), format="cif")
model = sf.get_model()
chain = model.get_chain(TARGET_CHAIN)

aa_mask = struc.filter_amino_acids(chain.atom_array)
target_aa_chain = Structure(chain.atom_array[aa_mask])

res_df = target_aa_chain.res_df()
assert not res_df.res_name_oneletter.isnull().any(), (
    f"Filtered chain {TARGET_CHAIN} still has non-AA "
    "residues - investigate the CIF"
)

target_sequence = "".join(res_df.res_name_oneletter)
print(f"Target: {TARGET_PDB_ID} chain {TARGET_CHAIN}, "
      f"{len(target_sequence)} aa")
print(f"  seq: {target_sequence[:60]}"
      f"{'...' if len(target_sequence) > 60 else ''}")

system = System([
    Protein(
        rep=target_sequence,
        id="target",
        structures={"1g13": target_aa_chain},
    ),
    Protein(
        rep=None,
        min_length=BINDER_MIN_LENGTH,
        max_length=BINDER_MAX_LENGTH,
        id="binder",
    ),
])

#  Attach an MSA to the target
MSA_CACHE = CACHE_DIR / f"{TARGET_PDB_ID.lower()}_{TARGET_CHAIN}.a3m"

if MSA_CACHE.exists():
    system[0].sequences = Sequences.from_file(MSA_CACHE, format="a3m")
    print(f"Loaded cached MSA from {MSA_CACHE}")
else:
    print("Querying ColabFold MMseqs2 server for target MSA...")
    try:
        # Returns a copy, so it must be reassigned. Done here,
        # before any .build(), so every system derived
        # downstream via with_instance_structures carries it.
        system = add_sequences_mmseqs2(system)
        hits = system[0].sequences.seqs
        with open(MSA_CACHE, "w") as fh:
            for i, s in enumerate(hits):
                fh.write(f">{s.id_ or f'seq_{i}'}\n{s.seq}\n")
        print(f"  -> cached {len(hits)} sequences at {MSA_CACHE}")
    except Exception as exc:
        print(
            f"  -> MSA lookup failed ({type(exc).__name__}: {exc}).\n"
        )

print(f"System: {len(system)} entities")
for i, entity in enumerate(system):
    rep_summary = (
        f"fixed ({len(entity.rep)} aa)"
        if entity.rep is not None
        else f"design ({entity.min_length}..{entity.max_length})"
    )
    has_struct = (
        "with structure"
        if entity.structures
        else "no structure"
    )
    n_msa = (
        len(entity.sequences.seqs)
        if entity.sequences is not None
        else 0
    )
    msa_summary = f"{n_msa} MSA seqs" if n_msa else "no MSA"
    print(f"  Entity {i} ({entity.id}): {rep_summary}, "
          f"{has_struct}, {msa_summary}")

## Step 2: Generate backbones with BoltzGen

Run BoltzGen's diffusion model to produce candidate binder backbones. The pipeline:
1. Writes a BoltzGen YAML spec for the system
2. Invokes the `boltzgen run` CLI
3. Parses outputs into `SystemInstance` objects with structures on `EntityInstance.models["model_0"]`

In [ ]:
generator = BoltzGenGenerator(
    protocol="protein-anything",
    device="cuda",
    num_devices=torch.cuda.device_count(),
    skip_inverse_folding=True,
    budget=10,
    keep_tmp_dir=True,
).build(system)

print(f"Generator ready: {generator.ready}")
print(f"Generating {NUM_BACKBONES} backbones")

# entities=[1]: design the binder, hold the 1G13 target fixed
backbones = generator.generate(num_designs=NUM_BACKBONES, entities=[1])

def _fmt(val):
    return f"{val:.3f}" if val is not None else "n/a"

print(f"Generated {len(backbones)} backbones:")
for i, bb in enumerate(backbones):
    meta = bb.metadata or {}
    metrics = meta.get("boltzgen_metrics") or {}
    rank = meta.get("boltzgen_rank")
    rank_str = f"rank={rank}, " if rank is not None else ""
    print(f"  Backbone {i}: {rank_str}"
          f"id={meta.get('boltzgen_design_id')}, "
          f"score={_fmt(bb.score)}, "
          f"confidence={_fmt(bb.confidence)}")
    binder_rep = bb[1].rep if bb[1].rep is not None else []
    if len(binder_rep) > 0:
        print(f"    binder seq: {''.join(binder_rep[:20])}... "
              f"(len={len(binder_rep)})")
    if metrics:
        print(f"    iptm={_fmt(metrics.get('iptm'))}, "
              f"ptm={_fmt(metrics.get('ptm'))}, "
              f"complex_plddt={_fmt(metrics.get('complex_plddt'))}")
    else:
        print(f"    (no metrics)")

## Step 3: Design sequences with LigandMPNN

In [ ]:
designed_instances = []

for bb_idx, backbone in enumerate(backbones):
    print(f"--- Backbone {bb_idx} -> MPNN ---")

    # Adapter: promote backbone.models["model_0"] onto Entity.structures for MPNN to consume
    bb_system = system.with_instance_structures(backbone)

    mpnn = LigandMPNN(
        model_name="ligandmpnn_v_32_010_25",
        device="cuda",
    ).build(bb_system)

    seqs = mpnn.generate(
        num_designs=NUM_SEQUENCES_PER_BACKBONE,
        temperature=0.1,
        entities=[1]
    )

    # Tag each design with its parent backbone for 
    # downstream tracing. LigandMPNN doesn't initialize 
    # .metadata so we set it to {} first if absent.
    for seq_idx, design in enumerate(seqs):
        if design.metadata is None:
            design.metadata = {}
        design.metadata["parent_backbone_idx"] = bb_idx
        design.metadata["parent_backbone_id"] = (
            backbone.metadata.get("boltzgen_design_id")
        )
        design.metadata["mpnn_seq_idx"] = seq_idx

    designed_instances.extend(seqs)
    print(f"  -> {len(seqs)} sequences designed")
    for d in seqs:
        print(f"    score={d.score:.3f}: {''.join(d[1].rep[:30])}...")

print(f"Total designed instances: {len(designed_instances)}")

In [ ]:
designed_instances[1]

## Step 4: Refold and validate with BoltzFold

Each MPNN-designed sequence is refolded by BoltzFold to confirm:
1. The sequence folds into a structure resembling the intended backbone
2. The complex still has high interface confidence (ipTM)

In [ ]:
from collections import defaultdict

def _fmt(val):
    return f"{val:.3f}" if val is not None else "n/a"

if len(designed_instances) == 0:
    raise RuntimeError(
        "No designed instances to refold. "
        "Re-run the MPNN step first."
    )

# Group designs by their parent backbone - all
# sequences from one backbone share the same binder
# length, so they can be batched through one
# BoltzFold.build()
by_backbone: dict[int, list] = defaultdict(list)
for inst in designed_instances:
    bb_idx = inst.metadata.get("parent_backbone_idx", -1)
    by_backbone[bb_idx].append(inst)

print(
    f"Refolding {len(designed_instances)} designs "
    f"across {len(by_backbone)} backbone groups "
    "(this may take several minutes)..."
)

refolded = []
for bb_idx in sorted(by_backbone.keys()):
    group = by_backbone[bb_idx]
    print(
        f"\n--- Backbone {bb_idx}: refolding "
        f"{len(group)} designs ---"
    )

    # Build a per-backbone template_system whose
    # binder length matches this group
    bb_template_system = system.with_instance_structures(
        group[0]
    )

    folder = BoltzFoldTransformer(
        device="cuda",
        sampling_steps=200,
        diffusion_samples=3,
        recycling_steps=3,
        use_msa=True,
        use_kernels=False,
    ).build(bb_template_system)

    group_refolded = folder.transform(group)
    refolded.extend(group_refolded)

print(f"\nRefolded {len(refolded)} designs total:")
for i, r in enumerate(refolded):
    parent = r.metadata.get("parent_backbone_idx", "?")
    score = _fmt(r.score) if r.score is not None else "n/a"
    conf = _fmt(r.confidence) if r.confidence is not None else "n/a"
    print(
        f"  Design {i} (from backbone {parent}): "
        f"score={score}, confidence={conf}"
    )

### Visualize BoltzFold refolds (aligned to parent backbone)

Each refold is **superimposed on its parent BoltzGen backbone** so you can see whether the MPNN-designed sequence folds back into the intended structure. Alignment is done on the shared **target** Cα atoms (a rigid Kabsch fit), and the same transform is applied to the binder so the whole complex moves together.

Coloring: target = grey, BoltzGen binder = blue, BoltzFold refold binder = orange. The closer the orange tracks the blue, the better the design reproduced. The printed **binder Cα RMSD (target-frame)** quantifies it.

In [ ]:
try:
    from py3Dmol import view
except ImportError:
    print("py3Dmol not installed. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "py3Dmol"])
    from py3Dmol import view

import tempfile
import os

def view_complex(system_instance, title="Structure"):
    """
    Visualize a protein structure using py3Dmol.
    
    Parameters
    ----------
    system_instance : SystemInstance
        The structure to visualize (contains EntityInstances with models)
    title : str
        Title for the visualization
    """
    # SystemInstance is list-like, containing EntityInstances
    # Each EntityInstance has a models dict with Structure objects
    
    if not system_instance or len(system_instance) == 0:
        print(f"{title}: Empty system instance")
        return
    
    # Get structures from all entities and combine them
    pdb_parts = []
    for entity_idx, entity in enumerate(system_instance):
        if hasattr(entity, 'models') and 'model_0' in entity.models:
            structure = entity.models['model_0']
            
            # Write to temporary file and read back as string
            with tempfile.NamedTemporaryFile(mode='w', suffix='.pdb', delete=False) as f:
                temp_path = f.name
            
            try:
                structure.to_file(temp_path, format="pdb")
                with open(temp_path, 'r') as f:
                    pdb_str = f.read()
                pdb_parts.append(pdb_str)
            finally:
                os.unlink(temp_path)
        else:
            print(f"{title}: Entity {entity_idx} has no model_0")
            return
    
    pdb_str = "".join(pdb_parts)
    
    if not pdb_str:
        print(f"{title}: No structure data available")
        return
    
    # Create py3Dmol viewer
    v = view(query='pdb data')
    v.addModel(pdb_str, "pdb")
    
    # Set cartoon style with spectrum coloring
    v.setStyle({}, {'cartoon': {'colorscheme': 'chainHetatm'}})
    
    v.zoomTo()
    
    # Display with title
    print(f"{title}")
    v.show()

# Test the function exists
print("✓ view_complex function with py3Dmol visualization ready")

In [ ]:
# Overlay each refold on its parent BoltzGen backbone
import tempfile
import os

import biotite.structure as struc
from py3Dmol import view

from evedesign.structure import Structure


def _entity_array(inst, idx):
    # Entity order in the System is (0) target, (1) binder.
    return inst[idx].models["model_0"].atom_array.copy()


def _ca(arr):
    return arr.coord[arr.atom_name == "CA"]


def _to_pdb(*arr_chain_pairs):
    parts = []
    for arr, chain in arr_chain_pairs:
        arr = arr.copy()
        arr.chain_id[:] = chain
        parts.append(arr)
    combined = Structure(struc.concatenate(parts))
    with tempfile.NamedTemporaryFile(mode="w", suffix=".pdb", delete=False) as f:
        path = f.name
    try:
        combined.to_file(path, format="pdb")
        with open(path) as fh:
            return fh.read()
    finally:
        os.unlink(path)


def view_aligned(backbone, refold, title="Aligned overlay", width=700, height=480):
    """Superimpose `refold` onto `backbone` on the shared target Cα and
    render the overlay. Returns a dict of Cα RMSDs (Å)."""
    bb_target, bb_binder = _entity_array(backbone, 0), _entity_array(backbone, 1)
    rf_target, rf_binder = _entity_array(refold, 0), _entity_array(refold, 1)

    metrics = {"target": None, "binder_target_frame": None, "binder_best_fit": None}
    bb_t, rf_t = _ca(bb_target), _ca(rf_target)
    if len(bb_t) > 0 and bb_t.shape == rf_t.shape:
        _, transform = struc.superimpose(bb_t, rf_t)
        rf_target = transform.apply(rf_target)
        rf_binder = transform.apply(rf_binder)
        metrics["target"] = struc.rmsd(_ca(bb_target), _ca(rf_target))
        bb_b, rf_b = _ca(bb_binder), _ca(rf_binder)
        if len(bb_b) > 0 and bb_b.shape == rf_b.shape:
            # binder RMSD in the target frame (fold + docking pose)
            metrics["binder_target_frame"] = struc.rmsd(bb_b, rf_b)
            # binder RMSD after independent best-fit (fold only)
            fitted_b, _ = struc.superimpose(bb_b, rf_b)
            metrics["binder_best_fit"] = struc.rmsd(bb_b, fitted_b)
    else:
        title += "  (target Cα mismatch - unaligned)"

    btf = metrics["binder_target_frame"]
    rmsd_str = f"{btf:.2f} A" if btf is not None else "n/a"
    print(f"{title}  |  binder Cα RMSD (target-frame): {rmsd_str}")

    bb_pdb = _to_pdb((bb_target, "A"), (bb_binder, "B"))
    rf_pdb = _to_pdb((rf_target, "C"), (rf_binder, "D"))
    v = view(width=width, height=height)
    v.addModel(bb_pdb, "pdb")   # model 0 = backbone
    v.addModel(rf_pdb, "pdb")   # model 1 = refold (aligned)
    v.setStyle({"model": 0, "chain": "A"}, {"cartoon": {"color": "lightgray"}})
    v.setStyle({"model": 0, "chain": "B"}, {"cartoon": {"color": "#3b6fb5"}})
    v.setStyle({"model": 1, "chain": "C"}, {"cartoon": {"color": "gray", "opacity": 0.35}})
    v.setStyle({"model": 1, "chain": "D"}, {"cartoon": {"color": "#e07a2b"}})
    v.zoomTo()
    v.show()
    return metrics


for i, r in enumerate(refolded):
    parent = r.metadata.get("parent_backbone_idx")
    title = (
        f"Refold {i} (from backbone {parent}): "
        f"score={r.score:.3f}, confidence={r.confidence:.3f}"
    )
    view_aligned(backbones[parent], r, title=title)

## Step 5: Filter and rank

Filter the refolded designs by structural confidence:
- `confidence > 0.7` (complex pLDDT - overall structure confidence)
- `score > 0.5` (the configured score_attribute, by default `confidence_score`)

Then sort by score descending. In production you'd also filter by interface metrics (ipTM, ipSAE) and liability counts.

In [ ]:
import pandas as pd

PLDDT_THRESHOLD = 0.7
SCORE_THRESHOLD = 0.5

rows = []
for i, r in enumerate(refolded):
    rows.append({
        "design_idx": i,
        "parent_backbone": r.metadata.get("parent_backbone_idx"),
        "parent_backbone_id": r.metadata.get("parent_backbone_id"),
        "score": r.score,
        "confidence": r.confidence,
        "sequence": "".join(r[1].rep) if r[1].rep is not None else "",
        "length": len(r[1].rep) if r[1].rep is not None else 0,
    })

df = pd.DataFrame(rows)
print("All designs:")
print(df.to_string(index=False))

passing = df[
    (df["confidence"] > PLDDT_THRESHOLD)
    & (df["score"] > SCORE_THRESHOLD)
].sort_values("score", ascending=False)

print(f"{len(passing)}/{len(df)} designs passed "
      f"(confidence > {PLDDT_THRESHOLD}, "
      f"score > {SCORE_THRESHOLD}):")
if len(passing) > 0:
    print(passing.to_string(index=False))
else:
    print("None, try lowering thresholds, increasing NUM_BACKBONES, or improving the target.")



## Overlay: BoltzGen backbone vs BoltzFold refold (aligned)

For the top-scoring refold, **superimpose** the BoltzFold refold onto its parent BoltzGen backbone and render both in a *single* viewer. We align on the **target** Cα atoms - the target is identical in both, so its residues correspond 1:1 - which puts both complexes in a common reference frame so the binders can be compared directly by eye.

Three Cα RMSDs are reported:
- **target** - residual after the alignment (the anchor; should be small).
- **binder (target-frame)** - binder deviation *after aligning the target*. Captures whether the binder both folds back **and** docks in the same pose.
- **binder (best-fit)** - binder deviation after independently superimposing the two binders. Isolates fold reproduction from pose.

A successful design has low values for both binder RMSDs and visibly overlapping binders (backbone binder in blue, refold binder in orange; the shared target is grey).

In [ ]:
# Reuses view_aligned defined above - same overlay, larger viewer,
# and the full RMSD breakdown for the single best design.
if len(refolded) == 0:
    print("No refolds to compare")
else:
    top_idx = max(range(len(refolded)), key=lambda i: refolded[i].score or 0.0)
    top_refold = refolded[top_idx]
    parent_idx = top_refold.metadata.get("parent_backbone_idx", 0)
    parent_backbone = backbones[parent_idx]

    print(f"Top refold: design {top_idx} (from backbone {parent_idx})")
    print(f"  backbone: score={parent_backbone.score:.3f}, confidence={parent_backbone.confidence:.3f}")
    print(f"  refold:   score={top_refold.score:.3f}, confidence={top_refold.confidence:.3f}\n")

    m = view_aligned(
        parent_backbone,
        top_refold,
        title=f"Top design {top_idx} (backbone {parent_idx})",
        width=900,
        height=600,
    )

    def _f(v):
        return f"{v:.2f} A" if v is not None else "n/a"

    print(f"\n  Ca RMSD  target (alignment anchor): {_f(m['target'])}")
    print(f"  Ca RMSD  binder (target-frame):     {_f(m['binder_target_frame'])}")
    print(f"  Ca RMSD  binder (best-fit):         {_f(m['binder_best_fit'])}")

## Export structures - backbone + best refold per backbone

Write CIF files to `./exports/`: for every BoltzGen backbone, the backbone complex itself plus that backbone's **best-scoring** BoltzFold refold (the top design after MPNN). Each complex is saved as a single CIF with the target on chain **A** and the binder on chain **B**, so you can open a backbone/refold pair in PyMOL/ChimeraX and superpose them.

In [ ]:
from pathlib import Path

import biotite.structure as struc

from evedesign.structure import Structure

# Where to write the CIFs
EXPORT_DIR = Path("./exports")
EXPORT_DIR.mkdir(exist_ok=True)

# Entity order in the System is (0) target, (1) binder -> chains A / B
CHAIN_BY_ENTITY = {0: "A", 1: "B"}


def save_complex_cif(system_instance, path):
    """Concatenate a SystemInstance's entities (model_0) into one
    structure with distinct chain IDs and write it as a CIF."""
    parts = []
    for idx, entity in enumerate(system_instance):
        arr = entity.models["model_0"].atom_array.copy()
        arr.chain_id[:] = CHAIN_BY_ENTITY.get(idx, chr(ord("A") + idx))
        parts.append(arr)
    Structure(struc.concatenate(parts)).to_file(str(path), format="cif")


# Best refold per parent backbone = highest score among its designs
best_refold_by_bb = {}
for r in refolded:
    bb = r.metadata.get("parent_backbone_idx")
    cur = best_refold_by_bb.get(bb)
    if cur is None or (r.score or 0.0) > (cur.score or 0.0):
        best_refold_by_bb[bb] = r

written = []
for bb_idx, backbone in enumerate(backbones):
    # 1) BoltzGen backbone complex
    bb_path = EXPORT_DIR / f"backbone_{bb_idx:02d}.cif"
    save_complex_cif(backbone, bb_path)
    written.append(bb_path)

    # 2) Best refold for this backbone (after MPNN -> BoltzFold)
    best = best_refold_by_bb.get(bb_idx)
    if best is None:
        print(f"backbone {bb_idx}: no refold found - skipping refold export")
        continue
    design_idx = refolded.index(best)
    rf_path = (
        EXPORT_DIR
        / f"backbone_{bb_idx:02d}_best_refold_design{design_idx:02d}"
        f"_score{best.score:.3f}.cif"
    )
    save_complex_cif(best, rf_path)
    written.append(rf_path)
    print(
        f"backbone {bb_idx}: backbone.cif + best refold "
        f"design {design_idx} (score={best.score:.3f}, "
        f"confidence={best.confidence:.3f}) -> {rf_path.name}"
    )

print(f"\nWrote {len(written)} CIF files to {EXPORT_DIR.resolve()}")